# Issue: solar/W medium, forecast horizon should be 80, but input data label's length is 28, leading to mismatch during evaluation

> Confirmed that the error happens with Toto's example notebook too

In [1]:
from ts_toolkit.pipeline.runner import TsPipeline
from ts_toolkit.pipeline.utils import load_config
import logging
from tqdm.notebook import tqdm
import pandas as pd
from dotenv import load_dotenv
import os
import numpy as np
from gluonts.model.forecast import QuantileForecast
import utilsforecast.processing as ufp
from gluonts.model import Forecast
from multivar_timeseries.evaluation.metrics import MAR, MBR

# auto reload imports in jupyter
%load_ext autoreload
%autoreload 2

/home/roc/01_projects/01_timeseries/timeseries-research/.venv/lib/python3.10/site-packages/fs/__init__.py:4: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  __import__("pkg_resources").declare_namespace(__name__)  # type: ignore


In [2]:
load_dotenv()
print(GIFT_EVAL_PATH := os.getenv("GIFT_EVAL"))
print(REPO_ROOT := os.getenv("REPO_ROOT"))

QUANTILE_LEVELS = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]

/home/roc/01_projects/01_timeseries/timeseries-research/data/gift_eval
/home/roc/01_projects/01_timeseries/timeseries-research


In [3]:
logging.basicConfig(level=logging.INFO)

# --- Load configuration ---
# config_path = "./configs/experiments/simpletime.yaml"
# config_path = "../configs/experiments/simpletime_sample.yaml"
config_path = "configs/experiments/personal/mar_mbr_debug.yaml"
# config_path = "./configs/experiments/gifteval.yaml"

config = load_config(config_path)

# --- Set up the evaluation pipeline ---
pipeline = TsPipeline(config)
# pipeline.run(skip_existing=False)

# pipeline.get_results()


Loading base config from <REPO_ROOT>/configs/experiments/personal/mar_mbr_debug.yaml...


INFO:root:   - Results will be saved to: /home/roc/01_projects/01_timeseries/timeseries-research/results/mar_mbr_debug_20251021-145925/ToTo-GiftEval-sample
INFO:root:   - Datasets will be loaded from: /home/roc/01_projects/01_timeseries/timeseries-research/data/gift_eval



In [4]:
config.experiment_name

'mar_mbr_debug_20251021-145925'

In [5]:
# import pandas as pd
# df_results = pd.read_csv(f"{REPO_ROOT}/results/{config.experiment_name}/{config.forecaster.alias}/all_results.csv")
# df_results

# Investigate

## Get Forecasts

In [6]:
import pandas as pd
import logging
from tqdm.notebook import tqdm
from ts_toolkit.models.ensembles.median import MedianEnsemble
from ts_toolkit.models import MULTIVARIATE_MODEL_LIST
from ts_toolkit.pipeline.gluonts_predictor import GluonTSPredictor
from ts_toolkit.pipeline.eval import Evaluator
from ts_toolkit.models import MODEL_REGISTRY, get_model
import os


In [7]:
dataset_cfg = pipeline.config.datasets[0]
to_univariate = pipeline.config.evaluation.to_univariate


In [8]:
evaluator = Evaluator(
    dataset_name=dataset_cfg.get("name"),
    term=dataset_cfg.get("term", "short"),
    to_univariate=to_univariate,
    # Create a subdirectory for each model's results
    output_path=f"{pipeline.output_path}/{pipeline.forecaster.alias}",
    storage_path=pipeline.storage_path,
    config=pipeline.config
)


INFO:ts_toolkit.pipeline.eval:Dataset 'solar/W' has target_dim=1 => to_univariate is forced set to False.


In [9]:
gluonts_dataset = evaluator.dataset.gluonts_dataset
prediction_length = evaluator.dataset.prediction_length
windows = evaluator.dataset.windows
prediction_length = evaluator.dataset.prediction_length

In [10]:
from ts_toolkit.dataloaders.base_loader import split

_, test_template = split(
    gluonts_dataset, offset=-prediction_length * windows
)
test_data = test_template.generate_instances(
    prediction_length=prediction_length,
    windows=windows,
    distance=prediction_length,
)


In [11]:
list(test_data.input)[0]['target'].__len__()

24

In [12]:
list(test_data)[0]

({'item_id': 'item_0',
  'start': Period('2005-12-31/2006-01-06', 'W-FRI'),
  'freq': 'W-FRI',
  'target': array([ 7068.2 , 10412.5 , 10468.4 ,  9614.2 ,  8804.7 , 13544.25,
         12863.85,  6893.3 , 14582.65, 13976.2 , 14010.8 , 15002.05,
         17603.4 , 16967.9 , 18647.4 , 16803.35, 15739.3 , 15946.75,
         14824.15, 19599.1 , 19512.55, 18987.85, 19022.25, 16205.25],
        dtype=float32)},
 {'item_id': 'item_0',
  'start': Period('2006-06-17/2006-06-23', 'W-FRI'),
  'freq': 'W-FRI',
  'target': array([19062.55, 16553.75, 16324.2 , 16541.95, 15334.  , 15490.15,
         14984.2 , 16178.95, 15478.8 , 13223.  , 13734.6 , 12097.05,
         12216.75, 13248.2 , 15811.05, 14933.65, 12849.85,  9569.  ,
          9536.85, 12201.9 , 12011.3 ,  9698.55, 12139.25,  9691.95,
         10796.05,  8602.5 ,  8667.  ,  8475.2 ], dtype=float32)})

In [13]:
test_data.label

LabelDataset(test_data=TestData(dataset=Map(fn=Compose(<gluonts.dataset.common.ProcessDataEntry object at 0x7a3dce937130>, <function itemize_start at 0x7a3eafb6e170>), iterable=Dataset({
    features: ['item_id', 'start', 'freq', 'target'],
    num_rows: 137
})), splitter=OffsetSplitter(offset=-80), prediction_length=80, windows=1, distance=80, max_history=None))

In [14]:
list(test_data.label)[0]['target'].__len__()

28

In [15]:
dataset = evaluator.dataset.test_data.input
dataset

InputDataset(test_data=TestData(dataset=Map(fn=Compose(<gluonts.dataset.common.ProcessDataEntry object at 0x7a3dce937130>, <function itemize_start at 0x7a3eafb6e170>), iterable=Dataset({
    features: ['item_id', 'start', 'freq', 'target'],
    num_rows: 137
})), splitter=OffsetSplitter(offset=-80), prediction_length=80, windows=1, distance=80, max_history=None))

In [16]:
list(evaluator.dataset.univar_test_data.input)[0]

{'item_id': 'item_0',
 'start': Period('2005-12-31/2006-01-06', 'W-FRI'),
 'freq': 'W-FRI',
 'target': array([ 7068.2 , 10412.5 , 10468.4 ,  9614.2 ,  8804.7 , 13544.25,
        12863.85,  6893.3 , 14582.65, 13976.2 , 14010.8 , 15002.05,
        17603.4 , 16967.9 , 18647.4 , 16803.35, 15739.3 , 15946.75,
        14824.15, 19599.1 , 19512.55, 18987.85, 19022.25, 16205.25],
       dtype=float32)}

In [17]:
h = pipeline.predictor.h or dataset.test_data.prediction_length
freq = pipeline.predictor.freq
print(h, freq, pipeline.predictor.batch_size)

80 None 1024


In [18]:
evaluator.forecasts = pipeline.predictor.predict(dataset)

100%|██████████| 3/3 [00:03<00:00,  1.03s/it]
/home/roc/01_projects/01_timeseries/timeseries-research/src/ts_toolkit/models/ensembles/median.py:131: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  fcst_df[self.alias] = _fcst_df[model_cols].median(axis=1)
/home/roc/01_projects/01_timeseries/timeseries-research/src/ts_toolkit/models/ensembles/median.py:139: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  fcst_df[q_col] = _fcst_df[models_q_cols].median(axis=1)
/home/roc/01_projects/01_timeseries/timeseries-researc

In [19]:
evaluator.forecasts.__len__(), [len(x.mean) for x in evaluator.forecasts][0]

(137, 80)

In [21]:
batch = []
for _, entry in enumerate(dataset):
    batch.append(entry)

# So batch is w * n_var * T
len(batch), [wind['target'].shape for wind in batch]

(137,
 [(24,),
  (24,),
  (24,),
  (24,),
  (24,),
  (24,),
  (24,),
  (24,),
  (24,),
  (24,),
  (24,),
  (24,),
  (24,),
  (24,),
  (24,),
  (24,),
  (24,),
  (24,),
  (24,),
  (24,),
  (24,),
  (24,),
  (24,),
  (24,),
  (24,),
  (24,),
  (24,),
  (24,),
  (24,),
  (24,),
  (24,),
  (24,),
  (24,),
  (24,),
  (24,),
  (24,),
  (24,),
  (24,),
  (24,),
  (24,),
  (24,),
  (24,),
  (24,),
  (24,),
  (24,),
  (24,),
  (24,),
  (24,),
  (24,),
  (24,),
  (24,),
  (24,),
  (24,),
  (24,),
  (24,),
  (24,),
  (24,),
  (24,),
  (24,),
  (24,),
  (24,),
  (24,),
  (24,),
  (24,),
  (24,),
  (24,),
  (24,),
  (24,),
  (24,),
  (24,),
  (24,),
  (24,),
  (24,),
  (24,),
  (24,),
  (24,),
  (24,),
  (24,),
  (24,),
  (24,),
  (24,),
  (24,),
  (24,),
  (24,),
  (24,),
  (24,),
  (24,),
  (24,),
  (24,),
  (24,),
  (24,),
  (24,),
  (24,),
  (24,),
  (24,),
  (24,),
  (24,),
  (24,),
  (24,),
  (24,),
  (24,),
  (24,),
  (24,),
  (24,),
  (24,),
  (24,),
  (24,),
  (24,),
  (24,),
  (24,),
  (2

In [22]:
# Extract all values for each variate across all windows
variates = batch[0]['target'].shape[0]  # number of variates
windows = len(batch)

# var_i: all values of variate i from all windows concatenated
var_dict = {}
for i in range(variates):
    var_dict[f'var_{i+1}'] = [batch[w]['target'][i].flatten() for w in range(windows)]
    var_dict[f'var_{i+1}'] = np.concatenate(var_dict[f'var_{i+1}'])

# var_i_j: all values of variate i from window j (j=1-based)
for i in range(variates):
    for j in range(windows):
        var_dict[f'var_{i+1}_{j+1}'] = batch[j]['target'][i].flatten()

# Example: var_1 contains all values of variate 1 from all windows
#          var_1_1 contains all values of variate 1 from window 1

# Check if the first 2376 values of all windows are the same for each variate
same_across_windows = {}
for i in range(variates):
    key = f'var_{i+1}_1'
    reference = var_dict[key][:2376]
    all_equal = True
    for j in range(2, windows+1):
        compare_key = f'var_{i+1}_{j}'
        if not np.array_equal(reference, var_dict[compare_key][:2376]):
            all_equal = False
            break
    same_across_windows[f'var_{i+1}'] = all_equal

same_across_windows

{'var_1': False,
 'var_2': False,
 'var_3': False,
 'var_4': False,
 'var_5': False,
 'var_6': False,
 'var_7': False,
 'var_8': False,
 'var_9': False,
 'var_10': False,
 'var_11': False,
 'var_12': False,
 'var_13': False,
 'var_14': False,
 'var_15': False,
 'var_16': False,
 'var_17': False,
 'var_18': False,
 'var_19': False,
 'var_20': False,
 'var_21': False,
 'var_22': False,
 'var_23': False,
 'var_24': False}

### Findings:
> 1. evaluator.dataset.test_data.input, which converts to the batch input, is in the shape of (n_windows, n_var, n_time_step), Each window and var contains all values of the previous window + 48 new time steps, which is the `h`
> 2. df is just all n_windows concatenated together vertically, each window with the same starting ds. So the number of rows of df = sum of all time steps in the (test) dataset. Multivariates are converted into y_i columns
> 3. predict_df takes df to run MedianEnsemble.forecast(), and returns fcst_df in the shape of (n_windows x n_var x h,  2 (unique_id, ds) + 1 (median) + 9 (quantiles)) or (6 x 7 x 48, 12)
> 4. Last bit of predict_df wrangles fcst_df to the shape of (n_windows x n_var, h) or (6 * 7, 48)

In [23]:
df = pipeline.predictor._gluonts_dataset_to_df(batch)
df.shape

(3288, 3)

In [24]:
df

,unique_id,ds,y
0,item_0-2006-06-17/2006-06-23,2006-01-06,7068.200195
1,item_0-2006-06-17/2006-06-23,2006-01-13,10412.500000
2,item_0-2006-06-17/2006-06-23,2006-01-20,10468.400391
3,item_0-2006-06-17/2006-06-23,2006-01-27,9614.200195
4,item_0-2006-06-17/2006-06-23,2006-02-03,8804.700195
...,...,...,...
3283,item_136-2006-06-17/2006-06-23,2006-05-19,9337.549805
3284,item_136-2006-06-17/2006-06-23,2006-05-26,10303.750000
3285,item_136-2006-06-17/2006-06-23,2006-06-02,9365.349609
3286,item_136-2006-06-17/2006-06-23,2006-06-09,10817.299805


In [25]:
metadata = pipeline.predictor.multivar_metadata if len(pipeline.predictor.target_cols) > 1 else pipeline.predictor.univar_metadata

In [26]:
fcst_df = pipeline.predictor.forecaster.forecast(
            df=df,
            h=h,
            freq=freq,
            level=pipeline.predictor.level,
            quantiles=pipeline.predictor.quantiles,
        )

100%|██████████| 3/3 [00:03<00:00,  1.04s/it]
/home/roc/01_projects/01_timeseries/timeseries-research/src/ts_toolkit/models/ensembles/median.py:131: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  fcst_df[self.alias] = _fcst_df[model_cols].median(axis=1)
/home/roc/01_projects/01_timeseries/timeseries-research/src/ts_toolkit/models/ensembles/median.py:139: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  fcst_df[q_col] = _fcst_df[models_q_cols].median(axis=1)
/home/roc/01_projects/01_timeseries/timeseries-researc

In [28]:
fcst_df

,unique_id,ds,ToTo-GiftEval-sample,ToTo-GiftEval-sample-q-10,ToTo-GiftEval-sample-q-20,ToTo-GiftEval-sample-q-30,ToTo-GiftEval-sample-q-40,ToTo-GiftEval-sample-q-50,ToTo-GiftEval-sample-q-60,ToTo-GiftEval-sample-q-70,ToTo-GiftEval-sample-q-80,ToTo-GiftEval-sample-q-90
0,item_0-2006-06-17/2006-06-23,2006-06-23,16044.858398,11113.873047,12882.148438,14113.361328,15359.929688,16044.858398,16495.107422,17547.789062,18726.392578,19930.072266
1,item_0-2006-06-17/2006-06-23,2006-06-30,15918.456055,11532.867188,13174.768555,14234.952148,14975.790039,15918.456055,16933.568359,17927.195312,18943.812500,20290.023438
2,item_0-2006-06-17/2006-06-23,2006-07-07,16155.234375,10888.682617,13072.801758,14487.498047,15762.063477,16155.234375,16814.791016,17958.730469,19062.705078,20376.517578
3,item_0-2006-06-17/2006-06-23,2006-07-14,16405.837891,12120.159180,13783.980469,14543.399414,15524.072266,16405.837891,17487.394531,18203.140625,19428.886719,21285.492188
4,item_0-2006-06-17/2006-06-23,2006-07-21,16780.429688,12308.314453,13328.469727,14611.052734,15554.807617,16780.429688,17592.132812,18462.951172,19520.677734,20881.158203
...,...,...,...,...,...,...,...,...,...,...,...,...
10955,item_99-2006-06-17/2006-06-23,2007-11-30,5787.226562,501.356293,2810.791260,4326.369141,5016.488281,5787.226562,6711.208008,7622.610352,8188.870117,9759.694336
10956,item_99-2006-06-17/2006-06-23,2007-12-07,4995.010742,1136.069214,2504.886475,3823.175781,4454.542969,4995.010742,5816.108887,6740.100098,7419.100098,8225.902344
10957,item_99-2006-06-17/2006-06-23,2007-12-14,5042.124023,367.135132,2365.456787,3560.988037,4296.487793,5042.124023,6261.630371,6780.443359,7529.978027,8432.905273
10958,item_99-2006-06-17/2006-06-23,2007-12-21,5934.073242,-1060.544556,1464.505615,3809.429688,4921.527832,5934.073242,6573.491211,7270.209473,7761.176270,8575.602539


In [29]:
fcst_df.shape

(10960, 12)

In [31]:
10960/80

137.0

In [32]:
fcst_df = fcst_df.set_index("unique_id")
fcsts: list[Forecast] = []
for uid, metadata_uid in tqdm(metadata.items(), total=len(metadata)):
    fcst_df_uid = fcst_df.loc[uid]
    forecast_arrays = ufp.value_cols_to_numpy(
        df=fcst_df_uid,
        id_col="unique_id",
        time_col="ds",
        target_col=None,
    )
    forecast_keys = ["mean"]
    if pipeline.predictor.quantiles is not None:
        forecast_keys += [f"{q}" for q in pipeline.predictor.quantiles]
    q_fcst = QuantileForecast(
        forecast_arrays=forecast_arrays.T,
        forecast_keys=forecast_keys,
        item_id=metadata_uid["item_id"],
        start_date=metadata_uid["fcst_start"],
    )
    fcsts.append(q_fcst)


  0%|          | 0/137 [00:00<?, ?it/s]

In [33]:
fcsts.__len__(), fcsts[0].quantile(0.5).__len__()

(137, 80)

In [34]:
fcsts_50qtl = [fcst.quantile(0.5) for fcst in fcsts]

In [25]:
# fcsts_arr = np.stack(fcsts)  # shape: (42, 48)

# # Reshape to (6, 7, h)
# fcsts_reshaped = fcsts_arr.reshape(6, 7, )
# fcsts_reshaped.shape

In [26]:
# fcsts_reshaped[0][0].quantile(0.5).shape

## Evaluate

### Evaluation Pipeline Summary

1. **Batching Data and Forecasts**  
   The original data and forecasts are passed to `gluonts.model.evaluation.evaluate_forecasts_raw`, where `data.input`, `data.label`, and forecasts are parsed and grouped into batches using a batcher.

2. **Batch Wrapping with ChainMap**  
   For each batch, the input, label, and forecasts are combined into a `ChainMap` object called `data_batch`, which is basically a merged dictionary for convenient access and is then passed to each metric (e.g., MAE, MSE) for evaluation. Here is why we need to modify the source code of `gluonts`, because in the original package `input` is only used to derive seasonality. The original `input` time series are not included in the final `data_batch`. 

3. **Metric Calculation Process**  
   - Each metric is instantiated as an evaluator, and the calculation is performed via `evaluator.update(data_batch)`.
   - The evaluator has two main components:
     - `stat`: Defines the error calculation (e.g., `absolute_error` for MAE). A `partial` wrapper is used to fix parameters such as `forecast_type='0.5'`.
     - `aggregate`: Typically a `Mean()` class with a specified axis for aggregation.
   - Under the hood, `evaluator.update` executes `aggregate.step(stat(data))`:
     - `stat(data)` computes the error for the batch (e.g., `absolute_error(data_batch, forecast_type='0.5')`).
     - `aggregate.step()` processes the error and updates:
       - `aggregate.partial_result`: The sum of all errors in the batch.
       - `aggregate.get()`: The mean error, calculated as `aggregate.partial_result / aggregate.n`.

4. **Aggregating Results Across Batches**  
   As the evaluation iterates through all batches, the evaluator accumulates the total error (`partial_result`) and the total number of time steps (`n`). The final average error across all batches is obtained via `aggregate.get()`.

### Notes
1. For multivar data, metrics are calculated with all vars flattened into one - aggregated over all time steps. Make sense when the number of steps among vars are largely different

### Preproc

In [35]:
from gluonts.ev.metrics import (
    MAE,
    MAPE,
    MASE,
    MSE,
    MSIS,
    ND,
    NRMSE,
    RMSE,
    SMAPE,
    MeanWeightedSumQuantileLoss,
)
from multivar_timeseries.evaluation.metrics import MAR, MBR, mean_abs_deviation_label, mean_abs_deviation_forecast, _get_input_means
from gluonts.ev.metrics import absolute_error
from gluonts.itertools import batcher
from gluonts.ev.aggregations import Mean

from gluonts.model.evaluation import _get_data_batch
from toolz import first

In [36]:
test_data = evaluator.dataset.test_data
# test_data = evaluator.dataset.univar_test_data
axis = None
metrics = [
    # MAE(),
    MAR(context_length=512)
]

batch_size=512 # ?

# forecasts = fcsts_reshaped
forecasts = fcsts


EVALUATOR = Evaluator(
    dataset_name=dataset_cfg.get("name"),
    term=dataset_cfg.get("term", "short"),
    to_univariate=to_univariate,
    # Create a subdirectory for each model's results
    output_path=f"{pipeline.output_path}/{pipeline.forecaster.alias}",
    storage_path=pipeline.storage_path,
    config=pipeline.config
)

seasonality = EVALUATOR.seasonality
mask_invalid_label = True
allow_nan_forecast = False

INFO:ts_toolkit.pipeline.eval:Dataset 'solar/W' has target_dim=1 => to_univariate is forced set to False.


In [37]:
label_ndim = first(test_data.label)["target"].ndim

assert label_ndim in [1, 2]

if axis is None:
    axis = tuple(range(label_ndim + 1))
if isinstance(axis, int):
    axis = (axis,)

assert all(ax in range(3) for ax in axis)

evaluators = {}
evaluator_metric = MAE()(axis=axis)
evaluators[evaluator_metric.name] = evaluator_metric


index_data = []

# TODO: Here the returned batch size is actually just one, within which there are several windows. 
# We may want to align the batcher and the windows, or remove batcher altogether.
input_batches = batcher(test_data.input, batch_size=batch_size)
label_batches = batcher(test_data.label, batch_size=batch_size)
forecast_batches = batcher(forecasts, batch_size=batch_size)

# pbar = tqdm()

input_batch, label_batch, forecast_batch = first(zip(input_batches, label_batches, forecast_batches))

if 0 not in axis:
    index_data.extend(
        [
            (forecast.item_id, forecast.start_date)
            for forecast in forecast_batch
        ]
    )

import logging
from collections import ChainMap
from typing import Iterable, List, Optional, Union
from dataclasses import dataclass
from toolz import first, valmap

import numpy as np
import pandas as pd
from tqdm import tqdm

from gluonts.dataset import DataEntry
from gluonts.dataset.split import TestData
from gluonts.ev.ts_stats import seasonal_error
from gluonts.itertools import batcher, prod
from gluonts.model import Forecast, Predictor
from gluonts.time_feature.seasonality import get_seasonality

from gluonts.model.evaluation import BatchForecast


data_batch = _get_data_batch(
            input_batch,
            label_batch,
            forecast_batch,
            seasonality=seasonality,
            mask_invalid_label=mask_invalid_label,
            allow_nan_forecast=allow_nan_forecast,
        )

### Metric Update

In [38]:
data_batch['label'].data.shape, data_batch['input'].__len__(), data_batch['input'][0].shape

((137, 28), 137, (24,))

In [31]:
mar = MAR(context_length=512)(axis=axis)
mar.update(data_batch)
mar.get()

1.507616267236432

In [32]:
mae = MAE()(axis=axis)
mae.update(data_batch)
mae.get()

ValueError: operands could not be broadcast together with shapes (137,28) (137,80) 